In [ ]:
# import patch_numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud # type: ignore
import spacy # type: ignore

In [ ]:
# %pip install spacy

In [ ]:
# !python -m spacy download en_core_web_sm

In [ ]:
url = "https://raw.githubusercontent.com/Jcharis/Bible-NLP-and-ML-using-Python/master/kjv_cleandata.csv"
data = pd.read_csv(url)
data.head()

In [ ]:
# extracting named entities
nlp = spacy.load("en_core_web_sm")

def extract_entities(text:str,entity_type: str = 'PERSON'):
  doc = nlp(text)
  entities = [ent.text for ent in doc.ents if ent.label_ == entity_type]
  return entities

In [ ]:
extract_entities("Jesse works with David in Accra","PERSON")

In [ ]:
data["ner_person"] = data["text"].apply(extract_entities)
data.head()

In [ ]:
person_list = [i for item in data['ner_person'].tolist() for i in item]
person_list

In [ ]:
def plot_wordcloud(docx):
  my_wordcloud = WordCloud().generate(docx)
  plt.imshow(my_wordcloud,interpolation="bilinear")
  plt.axis("off")
  plt.show()

In [ ]:
plot_wordcloud(" ".join(person_list))

In [ ]:
from collections import Counter
most_common_name = Counter(person_list)
most_common_name.most_common(20)

In [ ]:
bible_names_data = pd.DataFrame(most_common_name.most_common(20),columns=["names","count"])
bible_names_data.plot(kind="bar")

In [ ]:
# import plotly
# print("Plotly version: ", plotly.__version__)

In [ ]:
# %pip install --upgrade plotly

In [ ]:
# print("Plotly version: ", plotly.__version__)

In [ ]:
import plotly.express as px
px.bar(bible_names_data,x="names",y="count")

In [ ]:
# Network analysis
import networkx as nx

In [ ]:
def get_relationships(clean_data,window_size:int = 5, entity_column:str = 'person'):
  relationships = []
  
  for i in range(clean_data.index[-1]):
    end_i = min(i+5,clean_data.index[-1])
    char_list = sum((clean_data.loc[i:end_i][entity_column]),[])
    
    # Remove duplicated characters that are next to each other
    char_unique = [char_list[i] for i in range(len(char_list)) if (i==0) or char_list[i] != char_list[i-1]]
    
    if len(char_unique) > 1:
      for idx, a in enumerate(char_unique[:-1]):
        b = char_unique[idx + 1]
        relationships.append({"source":a, "target":b})
        
  return relationships 

In [ ]:
data["person"] = data['ner_person'].apply(lambda x: [item.split()[0] for item in x])

In [ ]:
clean_data = data

In [ ]:
relationship_data = get_relationships(clean_data)
relationship_data

In [ ]:
relationship_data = pd.DataFrame(relationship_data)
relationship_data.head()

In [ ]:
relationship_data = pd.DataFrame(np.sort(relationship_data.values,axis=1),columns=relationship_data.columns)
relationship_data

In [ ]:
relationship_data["value"] = 1
relationship_data = relationship_data.groupby(["source","target"],sort=False,as_index=False).sum()
relationship_data

In [ ]:
# creating a network graph
bible_graph = nx.from_pandas_edgelist(relationship_data,source="source",target="target",edge_attr="value",create_using=nx.Graph())
bible_graph

In [ ]:
nx.degree(bible_graph)

In [ ]:
nx.draw_networkx(bible_graph)

In [ ]:
# custom network plot
# from visualiser import GraphVisualization
import visualizer

In [ ]:
# import networkx as nx
from visualiser import GraphVisualization

# G = nx.karate_club_graph()
# pos = nx.spring_layout(G)

# vis = GraphVisualization(G, pos)
# fig = vis.create_figure(height=800, width=800, showlabel=True)
# fig.show()


In [ ]:
def plot_network(my_graph,**kwargs):
  default_kwargs = {
    "node_size":10,
    "node_border_width":1,
    "edge_width":0.5,
    "node_color":"blue"
  }
  
  default_kwargs.update(kwargs)
  
  pos = nx.spring_layout(my_graph,iterations=20)
  vis = GraphVisualization(my_graph,pos,**default_kwargs)
  fig = vis.create_figure(height=800,width=800,showlabel=False)
  return fig

In [ ]:
# plot_network(bible_graph)

In [ ]:

## Community Detection
communities = nx.community.louvain_communities(bible_graph)

In [ ]:
len(communities)

In [ ]:
# Function to create node color list based on communities
color_map_lists = ['Blues', 'BrBG', 'BuGn', 'BuPu', 'CMRmap', 'GnBu', 'Greens', 'Greys', 'OrRd', 'Oranges', 'PRGn', 'PiYG', 'PuBu', 'PuBuGn', 'PuOr', 'PuRd', 'Purples', 'RdBu', 'RdGy', 'RdPu', 'RdYlBu', 'RdYlGn', 'Reds', 'Spectral', 'Wistia', 'YlGn', 'YlGnBu', 'YlOrBr', 'YlOrRd', 'afmhot', 'autumn', 'binary', 'bone', 'brg', 'bwr', 'cool', 'coolwarm', 'copper', 'cubehelix', 'flag', 'gist_earth', 'gist_gray', 'gist_heat', 'gist_ncar', 'gist_rainbow', 'gist_stern', 'gist_yarg', 'gnuplot', 'gnuplot2', 'gray', 'hot', 'hsv', 'jet', 'nipy_spectral', 'ocean', 'pink', 'prism', 'rainbow', 'seismic', 'spring', 'summer', 'terrain', 'winter', 'Accent', 'Dark2', 'Paired', 'Pastel1', 'Pastel2', 'Set1', 'Set2', 'Set3', 'tab10', 'tab20', 'tab20b', 'tab20c']
color_map_list = ['aliceblue', 'antiquewhite', 'aqua', 'aquamarine', 'azure', 'beige', 'bisque', 'black', 'blanchedalmond', 'blue', 'blueviolet', 'brown', 'burlywood', 'cadetblue', 'chartreuse', 'chocolate', 'coral', 'cornflowerblue', 'cornsilk', 'crimson', 'cyan', 'darkblue', 'darkcyan', 'darkgoldenrod', 'darkgray', 'darkgrey', 'darkgreen', 'darkkhaki', 'darkmagenta', 'darkolivegreen', 'darkorange', 'darkorchid', 'darkred', 'darksalmon', 'darkseagreen', 'darkslateblue', 'darkslategray', 'darkslategrey', 'darkturquoise', 'darkviolet', 'deeppink', 'deepskyblue', 'dimgray', 'dimgrey', 'dodgerblue', 'firebrick', 'floralwhite', 'forestgreen', 'fuchsia', 'gainsboro', 'ghostwhite', 'gold', 'goldenrod', 'gray', 'grey', 'green', 'greenyellow', 'honeydew', 'hotpink', 'indianred', 'indigo', 'ivory', 'khaki', 'lavender', 'lavenderblush', 'lawngreen', 'lemonchiffon', 'lightblue', 'lightcoral', 'lightcyan', 'lightgoldenrodyellow', 'lightgray', 'lightgrey', 'lightgreen', 'lightpink', 'lightsalmon', 'lightseagreen', 'lightskyblue', 'lightslategray', 'lightslategrey', 'lightsteelblue', 'lightyellow', 'lime', 'limegreen', 'linen', 'magenta', 'maroon', 'mediumaquamarine', 'mediumblue', 'mediumorchid', 'mediumpurple', 'mediumseagreen', 'mediumslateblue', 'mediumspringgreen', 'mediumturquoise', 'mediumvioletred', 'midnightblue', 'mintcream', 'mistyrose', 'moccasin', 'navajowhite', 'navy', 'oldlace', 'olive', 'olivedrab', 'orange', 'orangered', 'orchid', 'palegoldenrod', 'palegreen', 'paleturquoise', 'palevioletred', 'papayawhip', 'peachpuff', 'peru', 'pink', 'plum', 'powderblue', 'purple', 'red', 'rosybrown', 'royalblue', 'rebeccapurple', 'saddlebrown', 'salmon', 'sandybrown', 'seagreen', 'seashell', 'sienna', 'silver', 'skyblue', 'slateblue', 'slategray', 'slategrey', 'snow', 'springgreen', 'steelblue', 'tan', 'teal', 'thistle', 'tomato', 'turquoise', 'violet', 'wheat', 'white', 'whitesmoke', 'yellow', 'yellowgreen']
def create_community_node_colors(graph, communities):
  number_of_colors = len(communities)
  node_colors = [color_map_list[i] for i in range(number_of_colors)]
  community_map = {node: i for i, comm in enumerate(communities) for node in comm}
  return [node_colors[community_map[node]] for node in graph.nodes()]

In [ ]:
node_colors = create_community_node_colors(bible_graph,communities)
# adding the community attribute to the graph
nx.set_node_attributes(bible_graph,communities,'group')
node_colors

In [ ]:
# plot_network(bible_graph,node_colors)

In [ ]:
node_degree = dict(bible_graph.degree)
node_degree

In [ ]:

# Find degree of centrality
degree_centrality = nx.degree_centrality(bible_graph)
degree_centrality

In [ ]:
degree_data = pd.DataFrame.from_dict(degree_centrality,orient="index",columns=["centrality"])
degree_data

In [ ]:
import plotly.express as px
degree_df = degree_data.sort_values('centrality', ascending=False)
fig2 = px.bar(degree_df.head(15),y="centrality", x=degree_df.head(15).index)
fig2.show()

In [ ]:
# What is the shortest connection between two people in the bible
nx.shortest_path(bible_graph,"David","Paul")

In [ ]:
# What is the shortest connection between two people in the bible
nx.shortest_path(bible_graph,"Joseph","Jesus")

In [ ]:
# What is the shortest connection between two people in the bible
nx.shortest_path(bible_graph,"David","Jesse")

In [ ]:
# saving the model to a file
import pickle

with open("bible_graph.pickle","wb") as f:
  pickle.dump(bible_graph,f)

In [ ]:

import pandas as pd
import numpy as np
import spacy
import networkx as nx

class SocialNetworkAnalyzer():
  def __init__(self, model="en_core_web_sm"):
    self.nlp = spacy.load(model)
  
  def extract_entities(self, text: str, entity_type: str = "PERSON"):
    doc = self.nlp(text)
    entities = [ent.text for ent in doc.ents if ent.label_ == entity_type]
    return entities
  
  def get_relationships(self, df, window_size: int = 5, entity_column: str = "person"):
    relationships = []

    for i in range(df.index[-1]):
      end_i = min(i+5, df.index[-1])
      char_list = sum((df.loc[i: end_i][entity_column]), [])
      
      char_unique = [char_list[i] for i in range(len(char_list)) 
                      if (i==0) or char_list[i] != char_list[i-1]]
      
      if len(char_unique) > 1:
        for idx, a in enumerate(char_unique[:-1]):
          b = char_unique[idx + 1]
          relationships.append({"source": a, "target": b})
    return relationships
  
  def analyze(self, df, entity_column="person"):
    df[entity_column] = df[f"ner_{entity_column}"].apply(lambda x: [item.split()[0] for item in x])
    relationships = self.get_relationships(df, entity_column=entity_column)
    relationship_df = pd.DataFrame(relationships)
    relationship_df = pd.DataFrame(np.sort(relationship_df.values, axis = 1), columns = relationship_df.columns)
    relationship_df["value"] = 1
    relationship_df = relationship_df.groupby(["source","target"], sort=False, as_index=False).sum()
    return relationship_df
  
  def create_network_graph(self, relationship_df):
    return nx.from_pandas_edgelist(relationship_df, source="source", target="target", edge_attr="value", create_using=nx.Graph())